# LeetCode 1971 — Find if Path Exists in Graph bằng Python

**Link đề:** https://leetcode.com/problems/find-if-path-exists-in-graph/
**Độ khó:** Easy · **Chủ đề:** Graph, Depth-First Search, Breadth-First Search, Union-Find

```python
class Solution:
    def validPath(self, n: int, edges: List[List[int]], source: int, destination: int) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu **đồ thị vô hướng** (undirected graph): đỉnh, cạnh, đường đi, **thành phần liên thông**.
2. Biết biểu diễn đồ thị bằng **danh sách kề** (adjacency list) với `list` hoặc `defaultdict(list)`.
3. Đi từ cách trực giác "lan truyền qua cạnh tới khi không đổi" → **BFS** (`collections.deque`) → **DFS** (stack / đệ quy) → **Union-Find**.
4. Hiểu vai trò của tập/mảng `visited` và vì sao thiếu nó sẽ lặp vô hạn.
5. Trace hàng đợi từng bước, chứng minh, Time/Space `O(V + E)`, lỗi hay gặp và biến thể.

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Có một đồ thị **hai chiều** (bi-directional) gồm `n` đỉnh đánh số `0 … n-1`. Các cạnh cho bởi mảng `edges`, mỗi `edges[i] = [u, v]` là cạnh nối `u` và `v` (đi được cả hai chiều). Không có cạnh trùng, không có cạnh tự nối.

Trả `True` nếu tồn tại **đường đi** (valid path) từ `source` tới `destination`, ngược lại `False`.

| `n` | `edges` | `source` | `destination` | Output |
|---:|---|:---:|:---:|:---:|
| 3 | `[[0,1],[1,2],[2,0]]` | 0 | 2 | `True` (0→2 hoặc 0→1→2) |
| 6 | `[[0,1],[0,2],[3,5],[5,4],[4,3]]` | 0 | 5 | `False` |

**Constraints:** `1 <= n <= 2·10^5`, `0 <= edges.length <= 2·10^5`, `0 <= u, v <= n-1`, `u != v`, `0 <= source, destination <= n-1`.

Chú ý: `edges` có thể **rỗng**; `source` có thể **bằng** `destination` (đường đi độ dài 0 → `True`). Kích thước tới `2·10^5` → cần thuật toán **tuyến tính** `O(n + E)`.

Hình dung ví dụ 2:

```text
  1 — 0 — 2          3 — 5
                      \  /
                       4
```

Hai "đảo" tách rời: `{0, 1, 2}` và `{3, 4, 5}`. Đỉnh `0` và `5` ở hai đảo khác nhau → không có đường đi. Mỗi "đảo" gọi là một **thành phần liên thông** (connected component). Bài toán thực chất hỏi: *`source` và `destination` có cùng thành phần liên thông không?*

## 2. Nền tảng Python và đồ thị

### 2.1. Danh sách kề (adjacency list)

Lưu cho **mỗi đỉnh** danh sách các đỉnh kề với nó. Với đồ thị vô hướng, cạnh `[u, v]` phải thêm **hai chiều**: `v` vào list của `u` và `u` vào list của `v`.

In [1]:
n = 6
edges = [[0, 1], [0, 2], [3, 5], [5, 4], [4, 3]]
graph = [[] for _ in range(n)]      # n list rỗng RIÊNG BIỆT
for u, v in edges:                   # tách cặp [u, v] thành hai biến (unpacking)
    graph[u].append(v)
    graph[v].append(u)
for node, neighbors in enumerate(graph):
    print(node, "->", neighbors)

0 -> [1, 2]
1 -> [0]
2 -> [0]
3 -> [5, 4]
4 -> [5, 3]
5 -> [3, 4]


### 2.2. Cái bẫy `[[]] * n`

`[[]] * n` tạo `n` tham chiếu tới **cùng một** list — thêm vào một ô là thêm vào tất cả. Phải dùng `[[] for _ in range(n)]`.

In [2]:
bad = [[]] * 3
bad[0].append(99)
print(bad)                        # [[99], [99], [99]] — cả 3 ô cùng thay đổi!
good = [[] for _ in range(3)]
good[0].append(99)
print(good)                       # [[99], [], []]

[[99], [99], [99]]
[[99], [], []]


### 2.3. `collections.defaultdict(list)`

`defaultdict(list)` là `dict` tự tạo `[]` khi truy cập key chưa có. Tiện khi số đỉnh không biết trước hoặc nhãn đỉnh không liên tiếp.

In [3]:
from collections import defaultdict

adj = defaultdict(list)
for u, v in [[0, 1], [1, 2]]:
    adj[u].append(v)
    adj[v].append(u)
print(dict(adj))
print(adj[7])          # [] — key mới được tạo tự động

{0: [1], 1: [0, 2], 2: [1]}
[]


### 2.4. `collections.deque` — hàng đợi hai đầu

BFS cần **hàng đợi** (queue): vào ở cuối, ra ở đầu (FIFO — first in, first out).
- `list.pop(0)` tốn `O(n)` vì phải dời mọi phần tử.
- `deque.popleft()` tốn `O(1)`.

In [4]:
from collections import deque

q = deque([0])
q.append(1)
q.append(2)
print(q)
print(q.popleft(), q)    # 0 ra trước — FIFO

deque([0, 1, 2])
0 deque([1, 2])


### 2.5. Stack bằng `list`

DFS lặp dùng **stack** (LIFO — last in, first out): `append` để đẩy vào, `pop()` (không đối số) để lấy phần tử **cuối** ra, đều `O(1)`.

In [5]:
stack = [0]
stack.append(1)
stack.append(2)
print(stack.pop(), stack)   # 2 ra trước — LIFO

2 [0, 1]


### 2.6. `visited` — đánh dấu đỉnh đã thăm

Đồ thị có **chu trình** (ví dụ 1: 0–1–2–0). Nếu không đánh dấu, ta sẽ đi vòng mãi. Dùng list bool `visited = [False] * n` (nhanh, đỉnh là số `0..n-1`) hoặc `set`.

## 3. Tính tay — "loang" từ `source`

Ví dụ 2, `source = 0`:

| Vòng | Tập đã tới được |
|---|---|
| bắt đầu | `{0}` |
| thêm hàng xóm của 0 | `{0, 1, 2}` |
| hàng xóm của 1, 2 | không có gì mới |

Tập dừng ở `{0, 1, 2}`, không chứa `5` → `False`. Đây là ý tưởng "**loang**" (flood fill) — nền tảng của BFS/DFS.

## 4. Cách 1 — Trực giác: lan truyền qua danh sách cạnh tới khi không đổi

Không cần dựng đồ thị: giữ tập `reach` (đã tới được). Lặp: duyệt **mọi cạnh**, nếu một đầu thuộc `reach` thì thêm đầu kia. Dừng khi một lượt duyệt không thêm được đỉnh nào.

In [6]:
def valid_path_relax(n, edges, source, destination):
    reach = {source}
    changed = True
    rounds = 0
    while changed:
        changed = False
        rounds += 1
        for u, v in edges:
            if u in reach and v not in reach:
                reach.add(v)
                changed = True
            elif v in reach and u not in reach:
                reach.add(u)
                changed = True
    return destination in reach

print(valid_path_relax(3, [[0, 1], [1, 2], [2, 0]], 0, 2))                    # True
print(valid_path_relax(6, [[0, 1], [0, 2], [3, 5], [5, 4], [4, 3]], 0, 5))    # False

True
False


### 4.1. Nút thắt của cách lan truyền

Mỗi lượt duyệt `E` cạnh; nếu các cạnh được liệt kê "ngược chiều" một đường dài, mỗi lượt chỉ tiến thêm **một** đỉnh → cần tới `~V` lượt → `O(V · E)`. Với `2·10^5` thì là `4·10^10` — quá chậm.

In [7]:
def rounds_needed(n, edges, source):
    reach, changed, rounds = {source}, True, 0
    while changed:
        changed, rounds = False, rounds + 1
        for u, v in edges:
            if (u in reach) != (v in reach):     # đúng một đầu đã tới được
                reach.update((u, v))
                changed = True
    return rounds

chain = 2000
edges_bad_order = [[i, i + 1] for i in range(chain - 2, -1, -1)]   # cạnh liệt kê từ cuối đường về đầu
edges_good_order = [[i, i + 1] for i in range(chain - 1)]
print("thứ tự xấu :", rounds_needed(chain, edges_bad_order, 0), "lượt")
print("thứ tự tốt :", rounds_needed(chain, edges_good_order, 0), "lượt")

thứ tự xấu : 2000 lượt
thứ tự tốt : 2 lượt


Vấn đề: mỗi lượt ta xem lại **mọi** cạnh, kể cả cạnh không liên quan tới đỉnh mới. Cải tiến: khi vừa tới một đỉnh, chỉ xem **các cạnh của chính đỉnh đó** — cần danh sách kề. Mỗi đỉnh được "mở rộng" đúng một lần → mỗi cạnh được xem 2 lần → `O(V + E)`. Đó là BFS/DFS.

## 5. Lời giải chính — BFS (Breadth-First Search), giảng từng đoạn

**Đoạn 1 — dựng danh sách kề** hai chiều.

**Đoạn 2 — khởi tạo:** `visited[source] = True`, hàng đợi `q = deque([source])`.

**Đoạn 3 — vòng lặp:** lấy `node = q.popleft()`; nếu `node == destination` → `True`. Với mỗi `nb` kề `node` chưa thăm: đánh dấu **ngay khi đưa vào hàng đợi**, rồi `append`.

**Đoạn 4:** hàng đợi rỗng mà chưa gặp `destination` → `False`.

In [8]:
def valid_path_bfs(n, edges, source, destination):
    graph = [[] for _ in range(n)]                # Đoạn 1
    for u, v in edges:
        graph[u].append(v)
        graph[v].append(u)

    visited = [False] * n                         # Đoạn 2
    visited[source] = True
    q = deque([source])

    while q:                                      # Đoạn 3
        node = q.popleft()
        if node == destination:
            return True
        for nb in graph[node]:
            if not visited[nb]:
                visited[nb] = True                # đánh dấu khi ĐƯA VÀO hàng đợi
                q.append(nb)
    return False                                  # Đoạn 4

print(valid_path_bfs(3, [[0, 1], [1, 2], [2, 0]], 0, 2))
print(valid_path_bfs(6, [[0, 1], [0, 2], [3, 5], [5, 4], [4, 3]], 0, 5))
print(valid_path_bfs(1, [], 0, 0))      # source == destination

True
False
True


**Vì sao đánh dấu khi đưa vào hàng đợi** (không phải khi lấy ra)? Nếu đợi tới lúc lấy ra mới đánh dấu, một đỉnh có thể được `append` **nhiều lần** bởi nhiều hàng xóm khác nhau trước khi tới lượt → hàng đợi phình to (vẫn đúng nhưng tốn bộ nhớ/thời gian, tệ nhất `O(E)` bản sao).

## 6. TRACE BFS từng bước

Đồ thị: `n = 6`, `edges = [[0,1],[0,3],[1,2],[3,4],[4,5]]`, `source = 0`, `destination = 5`.

```text
2 — 1 — 0 — 3 — 4 — 5
```

| Bước | popleft | Hàng xóm | Mới được đánh dấu | Hàng đợi sau |
|---:|:---:|---|---|---|
| 1 | 0 | 1, 3 | 1, 3 | `[1, 3]` |
| 2 | 1 | 0, 2 | 2 | `[3, 2]` |
| 3 | 3 | 0, 4 | 4 | `[2, 4]` |
| 4 | 2 | 1 | — | `[4]` |
| 5 | 4 | 3, 5 | 5 | `[5]` |
| 6 | 5 | — | — | gặp destination → `True` |

BFS đi theo **từng lớp khoảng cách**: lớp 0 `{0}`, lớp 1 `{1, 3}`, lớp 2 `{2, 4}`, lớp 3 `{5}`.

In [9]:
def trace_bfs(n, edges, source, destination):
    graph = [[] for _ in range(n)]
    for u, v in edges:
        graph[u].append(v)
        graph[v].append(u)
    visited = [False] * n
    visited[source] = True
    q = deque([source])
    step = 0
    while q:
        node = q.popleft()
        step += 1
        if node == destination:
            print(f"  bước {step}: lấy {node} = destination -> True")
            return True
        new = []
        for nb in graph[node]:
            if not visited[nb]:
                visited[nb] = True
                q.append(nb)
                new.append(nb)
        print(f"  bước {step}: lấy {node}, hàng xóm {graph[node]}, mới {new}, hàng đợi {list(q)}")
    print("  hàng đợi rỗng -> False")
    return False

print("Kết quả:", trace_bfs(6, [[0, 1], [0, 3], [1, 2], [3, 4], [4, 5]], 0, 5))
print()
print("Kết quả:", trace_bfs(6, [[0, 1], [0, 2], [3, 5], [5, 4], [4, 3]], 0, 5))

  bước 1: lấy 0, hàng xóm [1, 3], mới [1, 3], hàng đợi [1, 3]
  bước 2: lấy 1, hàng xóm [0, 2], mới [2], hàng đợi [3, 2]
  bước 3: lấy 3, hàng xóm [0, 4], mới [4], hàng đợi [2, 4]
  bước 4: lấy 2, hàng xóm [1], mới [], hàng đợi [4]
  bước 5: lấy 4, hàng xóm [3, 5], mới [5], hàng đợi [5]
  bước 6: lấy 5 = destination -> True
Kết quả: True

  bước 1: lấy 0, hàng xóm [1, 2], mới [1, 2], hàng đợi [1, 2]
  bước 2: lấy 1, hàng xóm [0], mới [], hàng đợi [2]
  bước 3: lấy 2, hàng xóm [0], mới [], hàng đợi []
  hàng đợi rỗng -> False
Kết quả: False


## 7. Cách 3 — DFS lặp bằng stack

Thay hàng đợi bằng stack → đi **sâu** theo một nhánh trước rồi mới quay lui. Với câu hỏi "có đường không", BFS và DFS đều đúng (chỉ khác thứ tự thăm).

In [10]:
def valid_path_dfs_iterative(n, edges, source, destination):
    graph = [[] for _ in range(n)]
    for u, v in edges:
        graph[u].append(v)
        graph[v].append(u)
    visited = [False] * n
    visited[source] = True
    stack = [source]
    order = []
    while stack:
        node = stack.pop()
        order.append(node)
        if node == destination:
            return True, order
        for nb in graph[node]:
            if not visited[nb]:
                visited[nb] = True
                stack.append(nb)
    return False, order

print(valid_path_dfs_iterative(6, [[0, 1], [0, 3], [1, 2], [3, 4], [4, 5]], 0, 5))   # thứ tự thăm khác BFS

(True, [0, 3, 4, 5])


### 7.1. Cách 4 — DFS đệ quy và giới hạn độ sâu

DFS đệ quy rất gọn, nhưng Python giới hạn độ sâu đệ quy (mặc định **1000**, tuỳ môi trường có thể khác — xem output bên dưới). Đồ thị là một đường dài `2·10^5` đỉnh sẽ gây `RecursionError`. Có thể nâng giới hạn bằng `sys.setrecursionlimit`, nhưng vẫn có nguy cơ tràn stack của C — vì vậy trên đồ thị lớn nên dùng **BFS/DFS lặp**.

In [11]:
import sys

def valid_path_dfs_recursive(n, edges, source, destination):
    graph = [[] for _ in range(n)]
    for u, v in edges:
        graph[u].append(v)
        graph[v].append(u)
    visited = [False] * n

    def dfs(node):
        if node == destination:
            return True
        visited[node] = True
        for nb in graph[node]:
            if not visited[nb] and dfs(nb):
                return True
        return False

    return dfs(source)

print(valid_path_dfs_recursive(3, [[0, 1], [1, 2], [2, 0]], 0, 2))
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())
long_chain = [[i, i + 1] for i in range(5000)]
try:
    valid_path_dfs_recursive(5001, long_chain, 0, 5000)
except RecursionError:
    print("RecursionError với đường dài 5001 đỉnh — dùng bản lặp thay thế")
print("BFS trên cùng đồ thị:", valid_path_bfs(5001, long_chain, 0, 5000))

True
giới hạn đệ quy mặc định: 1000
RecursionError với đường dài 5001 đỉnh — dùng bản lặp thay thế
BFS trên cùng đồ thị: True


## 8. Cách 5 — Union-Find (Disjoint Set Union)

Không cần duyệt đồ thị: coi mỗi đỉnh là một "nhóm" riêng. Mỗi cạnh `[u, v]` **gộp** nhóm của `u` và `v`. Cuối cùng hỏi `source` và `destination` có cùng nhóm không.

- `parent[x]`: "cha" của `x`; gốc của nhóm có `parent[r] == r`.
- `find(x)`: đi theo `parent` tới gốc. **Nén đường** (path compression): gán thẳng `parent` của các đỉnh trên đường về gốc (ở đây dùng kiểu "halving": `parent[x] = parent[parent[x]]`).
- `union(a, b)`: nối gốc của nhóm nhỏ vào gốc nhóm lớn (**union by size**) để cây thấp.

In [12]:
def valid_path_union_find(n, edges, source, destination):
    parent = list(range(n))
    size = [1] * n

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]   # nén đường (path halving)
            x = parent[x]
        return x

    for u, v in edges:
        ru, rv = find(u), find(v)
        if ru != rv:
            if size[ru] < size[rv]:
                ru, rv = rv, ru              # hoán đổi để ru là nhóm lớn hơn
            parent[rv] = ru
            size[ru] += size[rv]
    return find(source) == find(destination)

print(valid_path_union_find(3, [[0, 1], [1, 2], [2, 0]], 0, 2))
print(valid_path_union_find(6, [[0, 1], [0, 2], [3, 5], [5, 4], [4, 3]], 0, 5))

True
False


In [13]:
# Trace Union-Find: xem parent thay đổi sau từng cạnh
def trace_union_find(n, edges):
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x
    for u, v in edges:
        ru, rv = find(u), find(v)
        if ru != rv:
            parent[rv] = ru
        print(f"  cạnh {u}-{v}: gốc {ru} và {rv} -> parent = {parent}")
    groups = defaultdict(list)
    for x in range(n):
        groups[find(x)].append(x)
    print("  các nhóm:", list(groups.values()))

trace_union_find(6, [[0, 1], [0, 2], [3, 5], [5, 4], [4, 3]])

  cạnh 0-1: gốc 0 và 1 -> parent = [0, 0, 2, 3, 4, 5]
  cạnh 0-2: gốc 0 và 2 -> parent = [0, 0, 0, 3, 4, 5]
  cạnh 3-5: gốc 3 và 5 -> parent = [0, 0, 0, 3, 4, 3]
  cạnh 5-4: gốc 3 và 4 -> parent = [0, 0, 0, 3, 3, 3]
  cạnh 4-3: gốc 3 và 3 -> parent = [0, 0, 0, 3, 3, 3]
  các nhóm: [[0, 1, 2], [3, 4, 5]]


Union-Find đặc biệt hợp khi có **nhiều câu hỏi** "u và v có liên thông không" trên cùng đồ thị, hoặc khi cạnh **đến dần** (online).

## 9. Code nộp LeetCode (Python 3)

In [14]:
from collections import deque
from typing import List

class Solution:
    def validPath(self, n: int, edges: List[List[int]], source: int, destination: int) -> bool:
        if source == destination:
            return True
        graph = [[] for _ in range(n)]
        for u, v in edges:
            graph[u].append(v)
            graph[v].append(u)
        visited = [False] * n
        visited[source] = True
        q = deque([source])
        while q:
            node = q.popleft()
            for nb in graph[node]:
                if nb == destination:
                    return True           # dừng sớm ngay khi chạm destination
                if not visited[nb]:
                    visited[nb] = True
                    q.append(nb)
        return False


sol = Solution()
assert sol.validPath(3, [[0, 1], [1, 2], [2, 0]], 0, 2) is True
assert sol.validPath(6, [[0, 1], [0, 2], [3, 5], [5, 4], [4, 3]], 0, 5) is False
print("2 ví dụ của đề PASS")

2 ví dụ của đề PASS


## 10. Vì sao thuật toán đúng?

**Mệnh đề:** BFS từ `source` đánh dấu **đúng** tập các đỉnh tới được từ `source`.

- **Chỉ đánh dấu đỉnh tới được (an toàn):** một đỉnh `nb` được đánh dấu khi nó kề `node`, mà `node` đã được đánh dấu trước đó. Quy nạp theo thứ tự đánh dấu: `source` tới được; nếu `node` tới được và có cạnh `node–nb` thì `nb` tới được.
- **Đánh dấu mọi đỉnh tới được (đầy đủ):** giả sử có đường `source = x₀, x₁, …, x_k` mà `x_j` là đỉnh **đầu tiên** trên đường không được đánh dấu. Khi đó `x_{j-1}` được đánh dấu, nên nó vào hàng đợi và sau đó được `popleft` (hàng đợi chỉ rỗng khi mọi đỉnh đã vào đều đã ra). Lúc xử lý `x_{j-1}`, ta duyệt mọi hàng xóm, trong đó có `x_j` → `x_j` bị đánh dấu — mâu thuẫn.
- **Dừng:** mỗi đỉnh vào hàng đợi tối đa một lần (nhờ `visited`), nên vòng lặp kết thúc.

Do đó `destination` được gặp ⇔ có đường đi. Union-Find đúng vì sau khi gộp mọi cạnh, hai đỉnh cùng gốc ⇔ cùng thành phần liên thông.

## 11. Time / Space Complexity

Gọi `V = n`, `E = len(edges)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Lan truyền qua cạnh | `O(V · E)` tệ nhất | `O(V)` | trực giác, quá chậm |
| 2. BFS (chính) | `O(V + E)` | `O(V + E)` | an toàn, nên nộp |
| 3. DFS lặp | `O(V + E)` | `O(V + E)` | tương đương BFS |
| 4. DFS đệ quy | `O(V + E)` | `O(V + E)` + stack đệ quy `O(V)` | dễ `RecursionError` |
| 5. Union-Find (nén đường + theo kích thước) | `O(V + E · α(V))` | `O(V)` | α là hàm Ackermann ngược, coi như hằng số |

Dựng danh sách kề: `O(V + E)` (mỗi cạnh thêm 2 lần). BFS: mỗi đỉnh vào/ra hàng đợi một lần `O(V)`, mỗi danh sách kề được duyệt một lần `O(E)` tổng.

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.validPath(3, [[0, 1], [1, 2], [2, 0]], 0, 2), True)
check("đề 2", sol.validPath(6, [[0, 1], [0, 2], [3, 5], [5, 4], [4, 3]], 0, 5), False)
check("n = 1, không cạnh", sol.validPath(1, [], 0, 0), True)
check("source == destination, có cạnh", sol.validPath(3, [[0, 1]], 2, 2), True)
check("không cạnh, khác đỉnh", sol.validPath(2, [], 0, 1), False)
check("cạnh trực tiếp", sol.validPath(2, [[1, 0]], 0, 1), True)
check("chiều ngược của cạnh", sol.validPath(3, [[2, 1], [1, 0]], 0, 2), True)
check("đỉnh cô lập", sol.validPath(4, [[0, 1], [1, 2]], 0, 3), False)

PASS đề 1: True
PASS đề 2: False
PASS n = 1, không cạnh: True
PASS source == destination, có cạnh: True
PASS không cạnh, khác đỉnh: False
PASS cạnh trực tiếp: True
PASS chiều ngược của cạnh: True
PASS đỉnh cô lập: False


In [16]:
# Đối chiếu các cách trên đồ thị ngẫu nhiên có seed
import random
random.seed(1971)
funcs = [valid_path_relax, valid_path_bfs, lambda *a: valid_path_dfs_iterative(*a)[0],
         valid_path_dfs_recursive, valid_path_union_find, Solution().validPath]
trues = 0
for _ in range(400):
    n = random.randint(1, 12)
    all_pairs = [[u, v] for u in range(n) for v in range(u + 1, n)]
    edges = random.sample(all_pairs, random.randint(0, min(len(all_pairs), 10)))
    edges = [e if random.random() < 0.5 else e[::-1] for e in edges]   # đảo chiều ngẫu nhiên
    s, d = random.randrange(n), random.randrange(n)
    results = {f(n, edges, s, d) for f in funcs}
    assert len(results) == 1, (n, edges, s, d, results)
    trues += results.pop()
print("400 đồ thị ngẫu nhiên: mọi cách khớp; số ca True =", trues)

400 đồ thị ngẫu nhiên: mọi cách khớp; số ca True = 243


In [17]:
# Test quy mô lớn: đường thẳng 2*10^5 đỉnh (BFS và Union-Find không lỗi đệ quy)
N = 200000
big_edges = [[i, i + 1] for i in range(N - 1)]
random.shuffle(big_edges)
print(sol.validPath(N, big_edges, 0, N - 1), valid_path_union_find(N, big_edges, 0, N - 1))
cut = [e for e in big_edges if e != [N // 2, N // 2 + 1]]       # cắt đứt đường ở giữa
print(sol.validPath(N, cut, 0, N - 1), sol.validPath(N, cut, 0, N // 2))   # False True

True True


False True


## 13. Sai lầm người mới hay mắc

### Lỗi 1 — Chỉ thêm cạnh **một chiều**

In [18]:
def buggy_one_direction(n, edges, source, destination):
    graph = [[] for _ in range(n)]
    for u, v in edges:
        graph[u].append(v)          # QUÊN graph[v].append(u)
    visited = [False] * n
    visited[source] = True
    q = deque([source])
    while q:
        node = q.popleft()
        if node == destination:
            return True
        for nb in graph[node]:
            if not visited[nb]:
                visited[nb] = True
                q.append(nb)
    return False

print("buggy:", buggy_one_direction(3, [[2, 1], [1, 0]], 0, 2), "| đúng:", valid_path_bfs(3, [[2, 1], [1, 0]], 0, 2))

buggy: False | đúng: True


Đề nói **bi-directional** → mỗi cạnh phải thêm hai chiều.

### Lỗi 2 — Quên `visited` → lặp vô hạn trên chu trình

Bản dưới tự giới hạn số bước để minh hoạ thay vì treo máy thật.

In [19]:
def buggy_no_visited(n, edges, source, destination, max_steps=50):
    graph = [[] for _ in range(n)]
    for u, v in edges:
        graph[u].append(v)
        graph[v].append(u)
    q = deque([source])
    steps = 0
    while q and steps < max_steps:
        node = q.popleft()
        steps += 1
        if node == destination:
            return True, steps
        q.extend(graph[node])        # không kiểm tra đã thăm
    return "chưa dừng sau %d bước, hàng đợi còn %d phần tử" % (steps, len(q)), steps

print(buggy_no_visited(4, [[0, 1], [1, 2], [2, 0]], 0, 3))   # 3 cô lập, chu trình 0-1-2 làm hàng đợi phình mãi

('chưa dừng sau 50 bước, hàng đợi còn 51 phần tử', 50)


### Lỗi 3 — Dùng `[[]] * n` để tạo danh sách kề

Như mục 2.2: mọi đỉnh dùng chung một list → đồ thị "mọi đỉnh kề mọi đỉnh trong các cạnh", kết quả sai.

In [20]:
def buggy_shared_list(n, edges, source, destination):
    graph = [[]] * n                 # SAI
    for u, v in edges:
        graph[u].append(v)
        graph[v].append(u)
    return valid_path_bfs_given(graph, n, source, destination)

def valid_path_bfs_given(graph, n, source, destination):
    visited = [False] * n
    visited[source] = True
    q = deque([source])
    while q:
        node = q.popleft()
        if node == destination:
            return True
        for nb in graph[node]:
            if not visited[nb]:
                visited[nb] = True
                q.append(nb)
    return False

print("buggy:", buggy_shared_list(6, [[0, 1], [0, 2], [3, 5], [5, 4], [4, 3]], 0, 5), "| đúng: False")

buggy: True | đúng: False


### Lỗi 4 — Dùng `list.pop(0)` làm hàng đợi

Đúng kết quả nhưng mỗi `pop(0)` là `O(V)` → BFS thành `O(V²)`, với `2·10^5` đỉnh có thể quá thời gian. Dùng `deque.popleft()`.

### Lỗi 5 — Không xử lý `source == destination` khi kiểm tra ở hàng xóm

Code nộp ở mục 9 kiểm tra `nb == destination` khi duyệt hàng xóm (để dừng sớm). Nếu bỏ dòng `if source == destination: return True`, ca `n = 1, edges = []` sẽ trả `False` sai — vì `source` không bao giờ là "hàng xóm" của ai.

In [21]:
def buggy_no_same_check(n, edges, source, destination):
    graph = [[] for _ in range(n)]
    for u, v in edges:
        graph[u].append(v)
        graph[v].append(u)
    visited = [False] * n
    visited[source] = True
    q = deque([source])
    while q:
        node = q.popleft()
        for nb in graph[node]:
            if nb == destination:
                return True
            if not visited[nb]:
                visited[nb] = True
                q.append(nb)
    return False

print("buggy:", buggy_no_same_check(1, [], 0, 0), "| đúng:", sol.validPath(1, [], 0, 0))

buggy: False | đúng: True


## 14. Mẹo nhận dạng pattern

- "Có **đường đi** giữa hai đỉnh không", "hai điểm có **liên thông** không" → BFS / DFS / Union-Find.
- Cạnh cho dạng **danh sách cặp** → bước đầu tiên luôn là dựng **danh sách kề** (nhớ hai chiều nếu vô hướng).
- Đồ thị lớn (`10^5`) → dùng **lặp** (BFS/DFS stack), tránh đệ quy sâu; hàng đợi bằng `deque`.
- Nhiều truy vấn liên thông / cạnh thêm dần → **Union-Find**.
- Lưới 2D (ma trận) cũng là đồ thị: mỗi ô là đỉnh, kề 4 hướng — LeetCode 200 (Number of Islands), 733 (Flood Fill).
- Cùng họ: 547 (Number of Provinces), 323 (Number of Connected Components), 841 (Keys and Rooms), 1557.

## 15. Biến thể

### 15.1. Đếm số thành phần liên thông (LeetCode 323)

In [22]:
def count_components(n, edges):
    graph = [[] for _ in range(n)]
    for u, v in edges:
        graph[u].append(v)
        graph[v].append(u)
    visited = [False] * n
    components = 0
    for start in range(n):
        if visited[start]:
            continue
        components += 1                 # gặp đỉnh chưa thăm -> một thành phần mới
        visited[start] = True
        q = deque([start])
        while q:
            node = q.popleft()
            for nb in graph[node]:
                if not visited[nb]:
                    visited[nb] = True
                    q.append(nb)
    return components

print(count_components(6, [[0, 1], [0, 2], [3, 5], [5, 4], [4, 3]]))   # 2
print(count_components(5, []))                                         # 5

2
5


### 15.2. Độ dài đường đi ngắn nhất (số cạnh) — BFS lưu khoảng cách

Vì BFS đi theo lớp, khoảng cách lần đầu chạm một đỉnh chính là khoảng cách ngắn nhất (đồ thị không trọng số).

In [23]:
def shortest_path_length(n, edges, source, destination):
    graph = [[] for _ in range(n)]
    for u, v in edges:
        graph[u].append(v)
        graph[v].append(u)
    dist = [-1] * n                 # -1 = chưa thăm
    dist[source] = 0
    q = deque([source])
    while q:
        node = q.popleft()
        if node == destination:
            return dist[node]
        for nb in graph[node]:
            if dist[nb] == -1:
                dist[nb] = dist[node] + 1
                q.append(nb)
    return -1

print(shortest_path_length(3, [[0, 1], [1, 2], [2, 0]], 0, 2))                  # 1 (đi thẳng 0-2)
print(shortest_path_length(6, [[0, 1], [0, 3], [1, 2], [3, 4], [4, 5]], 0, 5))  # 3

1
3


### 15.3. In ra một đường đi cụ thể — lưu `prev`

In [24]:
def find_path(n, edges, source, destination):
    graph = [[] for _ in range(n)]
    for u, v in edges:
        graph[u].append(v)
        graph[v].append(u)
    prev = [None] * n
    seen = [False] * n
    seen[source] = True
    q = deque([source])
    while q:
        node = q.popleft()
        if node == destination:
            path = []
            while node is not None:
                path.append(node)
                node = prev[node]
            return path[::-1]
        for nb in graph[node]:
            if not seen[nb]:
                seen[nb] = True
                prev[nb] = node
                q.append(nb)
    return []

print(find_path(6, [[0, 1], [0, 3], [1, 2], [3, 4], [4, 5]], 0, 5))   # [0, 3, 4, 5]

[0, 3, 4, 5]


## 16. Bài tập (TODO — tự làm, không chạy tự động)

**Bài 1 (dự đoán output):** với `q = deque([1, 2]); q.append(3); q.popleft()` thì giá trị trả về và `q` còn lại là gì? Với `st = [1, 2]; st.append(3); st.pop()`?

**Bài 2 (tính tay):** `n = 7`, `edges = [[0,1],[1,2],[3,4],[5,6],[6,4]]`. Liệt kê các thành phần liên thông. Có đường `2 → 5` không? `3 → 5`?

**Bài 3 (điền chỗ trống):** hoàn thiện dựng danh sách kề hai chiều.

In [25]:
def exercise_build_graph(n, edges):
    graph = [[] for _ in range(n)]
    for u, v in edges:
        # TODO: thêm cạnh hai chiều
        pass
    return graph

# assert exercise_build_graph(3, [[0, 1], [1, 2]]) == [[1], [0, 2], [1]]

**Bài 4 (tự viết lại):** viết `exercise_valid_path(n, edges, source, destination)` bằng **DFS lặp** (stack), có `visited`.

In [26]:
def exercise_valid_path(n, edges, source, destination):
    # TODO
    pass

# assert exercise_valid_path(3, [[0, 1], [1, 2], [2, 0]], 0, 2) is True
# assert exercise_valid_path(6, [[0, 1], [0, 2], [3, 5], [5, 4], [4, 3]], 0, 5) is False
# assert exercise_valid_path(1, [], 0, 0) is True

**Bài 5 (biến thể):** viết `exercise_count_components(n, edges)` bằng **Union-Find** (số thành phần = số gốc khác nhau, hoặc `n` trừ số lần gộp thành công).

**Bài 6 (mini challenge):** viết `exercise_can_visit_all_rooms(rooms)` (LeetCode 841): phòng `0` mở sẵn; `rooms[i]` là danh sách chìa khoá trong phòng `i` (chìa `k` mở phòng `k`). Trả `True` nếu vào được mọi phòng. (Đây là đồ thị **có hướng**!)

In [27]:
def exercise_count_components(n, edges):
    # TODO
    pass

def exercise_can_visit_all_rooms(rooms):
    # TODO
    pass

# assert exercise_count_components(6, [[0, 1], [0, 2], [3, 5], [5, 4], [4, 3]]) == 2
# assert exercise_count_components(5, []) == 5
# assert exercise_can_visit_all_rooms([[1], [2], [3], []]) is True
# assert exercise_can_visit_all_rooms([[1, 3], [3, 0, 1], [2], [0]]) is False

## 17. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | FIFO vs LIFO | `popleft` lấy đầu, `pop` lấy cuối | `1, deque([2, 3])` và `3, [1, 2]` |
| 2 | loang từ từng đỉnh | `{0,1,2}`, `{3,4,5,6}` | 2→5 không; 3→5 có (3-4-6-5) |
| 3 | vô hướng | hai lệnh `append` | `graph[u].append(v); graph[v].append(u)` |
| 4 | thay deque bằng list | `stack.pop()` | đánh dấu khi `append` vào stack |
| 5 | mỗi lần gộp hai gốc khác nhau giảm 1 thành phần | `parent = list(range(n))` | `components = n`; `if ru != rv: parent[rv] = ru; components -= 1` |
| 6 | đồ thị có hướng `i → k` | BFS/DFS từ phòng 0, **một chiều** | so `sum(visited) == len(rooms)` |

## 18. Đáp án tham khảo — chỉ xem sau khi tự làm

**Bài 1:** `popleft()` trả `1`, còn `deque([2, 3])`; `pop()` trả `3`, còn `[1, 2]`.

**Bài 2:** thành phần `{0, 1, 2}` và `{3, 4, 5, 6}`; `2 → 5`: không; `3 → 5`: có (`3–4–6–5`).

In [28]:
q = deque([1, 2]); q.append(3); print(q.popleft(), q)
st = [1, 2]; st.append(3); print(st.pop(), st)
print(valid_path_bfs(7, [[0, 1], [1, 2], [3, 4], [5, 6], [6, 4]], 2, 5),
      valid_path_bfs(7, [[0, 1], [1, 2], [3, 4], [5, 6], [6, 4]], 3, 5),
      find_path(7, [[0, 1], [1, 2], [3, 4], [5, 6], [6, 4]], 3, 5))


def answer_build_graph(n, edges):
    graph = [[] for _ in range(n)]
    for u, v in edges:
        graph[u].append(v)
        graph[v].append(u)
    return graph


def answer_valid_path(n, edges, source, destination):
    graph = answer_build_graph(n, edges)
    visited = [False] * n
    visited[source] = True
    stack = [source]
    while stack:
        node = stack.pop()
        if node == destination:
            return True
        for nb in graph[node]:
            if not visited[nb]:
                visited[nb] = True
                stack.append(nb)
    return False


def answer_count_components(n, edges):
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    components = n
    for u, v in edges:
        ru, rv = find(u), find(v)
        if ru != rv:
            parent[rv] = ru
            components -= 1
    return components


def answer_can_visit_all_rooms(rooms):
    visited = [False] * len(rooms)
    visited[0] = True
    stack = [0]
    while stack:
        room = stack.pop()
        for key in rooms[room]:
            if not visited[key]:
                visited[key] = True
                stack.append(key)
    return all(visited)


assert answer_build_graph(3, [[0, 1], [1, 2]]) == [[1], [0, 2], [1]]
assert answer_valid_path(3, [[0, 1], [1, 2], [2, 0]], 0, 2) is True
assert answer_valid_path(6, [[0, 1], [0, 2], [3, 5], [5, 4], [4, 3]], 0, 5) is False
assert answer_valid_path(1, [], 0, 0) is True
assert answer_count_components(6, [[0, 1], [0, 2], [3, 5], [5, 4], [4, 3]]) == 2
assert answer_count_components(5, []) == 5
assert answer_can_visit_all_rooms([[1], [2], [3], []]) is True
assert answer_can_visit_all_rooms([[1, 3], [3, 0, 1], [2], [0]]) is False
for _ in range(200):
    n = random.randint(1, 10)
    pairs = [[u, v] for u in range(n) for v in range(u + 1, n)]
    edges = random.sample(pairs, random.randint(0, min(len(pairs), 8)))
    assert answer_count_components(n, edges) == count_components(n, edges)
print("Đáp án tham khảo PASS toàn bộ test!")

1 deque([2, 3])
3 [1, 2]
False True [3, 4, 6, 5]
Đáp án tham khảo PASS toàn bộ test!


## 19. Template nhớ nhanh + checklist submit

```python
from collections import deque

class Solution:
    def validPath(self, n, edges, source, destination):
        graph = [[] for _ in range(n)]
        for u, v in edges:
            graph[u].append(v); graph[v].append(u)     # hai chiều
        visited = [False] * n
        visited[source] = True
        q = deque([source])
        while q:
            node = q.popleft()
            if node == destination:
                return True
            for nb in graph[node]:
                if not visited[nb]:
                    visited[nb] = True                 # đánh dấu khi đưa vào
                    q.append(nb)
        return False
```

**Câu thần chú:** *dựng danh sách kề hai chiều → loang từ source bằng deque + visited → chạm destination là True.*

**Checklist trước khi submit:**
- [ ] Thêm cạnh **hai chiều**?
- [ ] Danh sách kề tạo bằng `[[] for _ in range(n)]`, không phải `[[]] * n`?
- [ ] Có `visited`, đánh dấu **khi đưa vào** hàng đợi/stack?
- [ ] Dùng `deque.popleft()` (không `list.pop(0)`), không đệ quy sâu với `n = 2·10^5`?
- [ ] `source == destination` (kể cả `edges` rỗng) trả `True`?